# Hierarchischer Ansatz: MLP

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Ein kleines MLP je Stufe im hierarchischen Ansatz testen (#102).

In [ ]:
%load_ext autoreload
%autoreload 2

from sklearn.neural_network import MLPClassifier

from awp2.config import MLP_MAX_ITER, MODEL_DOCS_FIGURES_DIR, SEED
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import HierarchicalClassifier, summarize_hierarchical, summary_table
from awp2.models.tuned import refit_untracked, tuned_hierarchical_svm
from awp2.plots import plot_error_breakdown, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Hypothese

Ein kleines neuronales Netz (MLP) je Stufe ist der erste NN-Vergleich im hierarchischen Ansatz. Es kann nichtlineare Grenzen lernen wie die RBF-SVM, braucht aber mehr Daten. Frühes Stoppen begrenzt die Überanpassung, ausgeglichene Gewichte (`balance_samples=True`, je Stufe umskaliert) die Klassenungleichheit.

Gleicher Ablauf wie bei der Referenz `hierarchical_svm` (#87): CV-Suche mit `tune()` auf dem
Trainingsteil, dann genau ein `run()`. Verglichen wird per CV mit der SVM-Referenz, danach
Fehlerzerlegung, Oracle-Kultur und McNemar gegen die SVM auf der Validierung.

In [ ]:
reference = tuned_hierarchical_svm()
svm_cv_score = reference.cv_score
preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
TOP_CANDIDATES = 15


def mlp() -> MLPClassifier:
    return MLPClassifier(early_stopping=True, max_iter=MLP_MAX_ITER, random_state=SEED)


model = HierarchicalClassifier(mlp(), mlp())

hidden = [(128, 64), (256, 128)]
alpha = [0.01, 0.1]
param_grid = {
    "crop_estimator__hidden_layer_sizes": hidden,
    "crop_estimator__alpha": alpha,
    "stage_estimator__hidden_layer_sizes": hidden,
    "stage_estimator__alpha": alpha,
}

## CV-Suche

In [ ]:
tuned = tune(
    model,
    TuneConfig(
        name="hierarchical_mlp_search",
        approach="hierarchical",
        description="MLP je Stufe im hierarchischen Ansatz (#102).",
        preprocessing=preprocessing,
        balance_samples=True,
        param_grid=param_grid,
    ),
)
tuned.best_params, round(tuned.best_score, 4), round(tuned.best_score - svm_cv_score, 4)

In [ ]:
tuning_ax = plot_tuning_candidates(
    tuned.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="crop_estimator__alpha"
)
tuning_ax.axvline(svm_cv_score, color="black", linestyle="--", linewidth=1)
tuning_ax.set_title("Hierarchisch MLP: CV-Kandidaten (gestrichelt: SVM-Referenz)")
save_doc_figure(tuning_ax.figure, "hierarchical_mlp_tuning", MODEL_DOCS_FIGURES_DIR)

## Einmalige Bewertung

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="hierarchical_mlp",
        approach="hierarchical",
        description="Bester MLP aus hierarchical_mlp_search, einmal bewertet.",
        preprocessing=preprocessing,
        balance_samples=True,
        tuning_run=tuned.run_id,
    ),
)
result.metrics.model_dump(), result.run_id

## Vergleich mit der SVM-Referenz

Die SVM-Referenz (`tuned_hierarchical_svm()`) wird ohne MLflow nachtrainiert
(`refit_untracked`), um die Vorhersagen Zeile für Zeile zu vergleichen.

In [ ]:
svm_result = refit_untracked(reference, "hierarchical_svm_refit")
summaries = {
    "RBF-SVM (Referenz)": summarize_hierarchical(svm_result.pipeline, split.X_val, split.y_val),
    "MLP": summarize_hierarchical(result.pipeline, split.X_val, split.y_val),
}
breakdowns = {label: summary.breakdown for label, summary in summaries.items()}
(
    summary_table(summaries).round(3),
    compare_predictions(split.y_val, svm_result.y_pred, result.y_pred),
)

In [ ]:
breakdown_ax = plot_error_breakdown(breakdowns)
breakdown_ax.set_title("Fehlerzerlegung: MLP vs. RBF-SVM")
save_doc_figure(breakdown_ax.figure, "hierarchical_mlp_breakdown", MODEL_DOCS_FIGURES_DIR)

## Befund

- CV 0.789, Validierung 0.787 – nach CV auf Platz 4 der Serie, deutlich unter der SVM
  (0.839 / 0.844).
- Größere Netze (256, 128) gewinnen; alle gewählten Werte liegen am Rand des kleinen Rasters.
- Verluste auf beiden Stufen: 13 % falsche Kultur, 8 % nur Stadium falsch (SVM 9 % / 4 %).